# Part 4 — Replay & Trace Walkthrough (live example)

This notebook demonstrates a concrete, end-to-end reproducibility workflow:

1. **Run #1 (vendor)**: start a trace, call a tool (with a fake vendor), write cache.
2. **Run #2 (replay)**: call the same tool again → served from **cache**.
3. **"New machine" replay**: copy the cache DB, disable vendor calls, confirm cache still serves.

We use a **fake Polygon client** so this notebook is runnable without network/API keys.

## Run with the project interpreter
Recommended:
```bash
poetry run jupyter lab
```


## 0) Imports + ensure `src/` is importable

If you're running from repo root, adding `./src` to `sys.path` makes `voli` importable.


In [ ]:
import os
import shutil
import sys
from pathlib import Path
from tempfile import TemporaryDirectory

repo_root = Path().resolve()
src_path = repo_root / "src"
if src_path.exists() and str(src_path) not in sys.path:
    sys.path.insert(0, str(src_path))

In [ ]:
from voli.run_trace import end_trace, start_trace
from voli.tool_schemas import GetOptionQuotesInput
from voli.tools import polygon_tools as pt

print("python:", sys.executable)
print("repo_root:", repo_root)

## 1) Isolate cache + trace outputs

We’ll write cache + traces into a temporary directory so you can run this notebook repeatedly without touching `~/.voli/`.


In [ ]:
tmp = TemporaryDirectory()
tmp_path = Path(tmp.name)

cache_path = tmp_path / "cache.sqlite"
trace_dir = tmp_path / "traces"

os.environ["VOLI_CACHE_PATH"] = str(cache_path)
os.environ["VOLI_TRACE_DIR"] = str(trace_dir)

# IMPORTANT: clear lru_cache so tools pick up the new env var
pt._get_cache.cache_clear()

print("cache_path:", cache_path)
print("trace_dir :", trace_dir)

## 2) Install a fake Polygon client (no network)

We patch `pt.PolygonClient` to a fake implementation that returns deterministic snapshots.
We also track how many vendor calls happen.


In [ ]:
vendor_calls = []


class FakePolygonClient:
    def get_option_contract_snapshot(self, underlying: str, sym: str) -> dict:
        vendor_calls.append((underlying, sym))
        ts_ns = 1700000000000000000
        return {
            "results": {
                "details": {"ticker": sym},
                "last_quote": {
                    "bid": 1.0,
                    "ask": 1.1,
                    "sip_timestamp": ts_ns,
                    "participant_timestamp": ts_ns,
                },
                "last_trade": {"price": 1.05, "sip_timestamp": ts_ns},
                "last_updated": ts_ns,
            }
        }

    def close(self) -> None:
        return None


# patch
pt.PolygonClient = FakePolygonClient
print("FakePolygonClient installed")

## 3) Run #1 (vendor) — start trace, call tool, write cache

We call `get_option_quotes` with duplicate symbols to prove:
- vendor calls are deduped per-request
- output preserves original input multiplicity/order


In [ ]:
s1 = "O:NVDA251219C00100000"
s2 = "O:NVDA251219P00100000"
inp = GetOptionQuotesInput(option_symbols=[s1, s2, s1], asof=None)

t1 = start_trace("demo_run_1")
out1 = pt.get_option_quotes(inp)
end_trace()

print("primary_source:", out1.meta.primary_source)
print("warnings      :", out1.meta.warnings)
print("vendor_calls  :", len(vendor_calls), vendor_calls)
print("trace_file    :", t1.path)
print("quotes order  :", [q.option_symbol for q in out1.quotes])

### Inspect the trace file (run #1)
You should see `tool_call` with `primary_source='polygon'`.


In [ ]:
print(t1.path.read_text(encoding="utf-8"))

## 4) Run #2 (replay) — same inputs, served from cache

Expected:
- `primary_source='cache'`
- vendor call count does not increase


In [ ]:
t2 = start_trace("demo_run_2")
out2 = pt.get_option_quotes(inp)
end_trace()

print("primary_source:", out2.meta.primary_source)
print("warnings      :", out2.meta.warnings)
print("vendor_calls  :", len(vendor_calls), vendor_calls)
print("trace_file    :", t2.path)
print("quotes order  :", [q.option_symbol for q in out2.quotes])

### Inspect the trace file (run #2)
You should see `tool_call` with `primary_source='cache'`.


In [ ]:
print(t2.path.read_text(encoding="utf-8"))

## 5) "New machine" replay

We simulate a new environment by:
1) copying the cache DB
2) pointing `VOLI_CACHE_PATH` at the copy
3) replacing the vendor client with one that **raises** if called

If replay works, tool calls should still succeed and serve from **cache**.


In [ ]:
replay_cache = tmp_path / "replay-cache.sqlite"
shutil.copy2(cache_path, replay_cache)
print("copied cache to:", replay_cache)

os.environ["VOLI_CACHE_PATH"] = str(replay_cache)
pt._get_cache.cache_clear()


class ExplodingPolygonClient:
    def get_option_contract_snapshot(self, underlying: str, sym: str) -> dict:
        raise RuntimeError("Vendor should not be called during replay!")

    def close(self) -> None:
        return None


pt.PolygonClient = ExplodingPolygonClient

t3 = start_trace("demo_run_3_new_machine")
out3 = pt.get_option_quotes(inp)
end_trace()

print("primary_source:", out3.meta.primary_source)
print("trace_file    :", t3.path)
print("quotes order  :", [q.option_symbol for q in out3.quotes])

## 6) Cleanup
The temp directory is cleaned up when this kernel exits, but you can close it explicitly.


In [ ]:
tmp.cleanup()
print("cleaned up temp dir")